In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = "../../helpers/lng_lnd"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = "../../helpers/curated_helpers"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_assignment_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "AssignmentId String,"
        #     "SourceSystem String,"
        #     "PortfolioKey String,"
        #     "PortfolioName String,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId String,"
        #     "AssignmentKey String,"
        #     "CaseCode Int,"
        #     "ExposureKey String,"
        #     "PartyKey String, "
        #     "CDI String,"
        #     "TaxIdentificationNumber String,"
        #     "AgreementKey String,"
        #     "InternalAssignmentStartDate String,"
        #     "ExternalAssignmentStartDate String,"
        #     "AssignmentRecallDate timestamp,"
        #     "ChannelCode String,"
        #     "Channel String,"
        #     "ExternalCompanyAgency String,"
        #     "ExternalCompanyPartyKey String,"
        #     "ExternalCompanyFullName String,"
        #     "AssignmentTypeCode String,"
        #     "AssignmentType String,"
        #     "TeamCode String,"
        #     "Team String,"
        #     "AgentPartyKey String,"
        #     "AgentFullName String,"
        #     "AgentLoginName String,"
        #      "Supervisor String,"
        #      "AssistantManager String,"
        #      "Manager String,"
        #      "Head String,"
        #     "AssignmentBucketCode Int,"
        #     "AssignmentBucket String,"
        #     "AssignmentProcess String,"
        #     "ProcessType String,"
        #     "PacketCode Int,"
        #     "Packet String,"
        #     "PacketPreviousEndDate timestamp,"
        #     "PacketAssignmentDate timestamp,"
        #     "PaccCode Int,"
        #     "QueCode Int,"
        #     "Que String,"
        #     "Segment String,"
        #     "ExtraSegment String,"
        #     "StreamTypeCode Int,"
        #     "StreamType String,"
        #     "StreamStatusCode Int,"
        #     "StreamStatus String,"
        #     "StreamAssignedStatusCode Int,"
        #     "StreamAssignedStatus String,"
        #     "OutstandingAmount double,"
        #     "DebtAmount double,"
        #     "BusinessDate Date,"
        #    )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Assignment"

        # ================== Create Delta Table ==================

        # logging.info("Creating Delta Table...")
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")


        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== arxAssign ==================

        arxAssign_extra_filters = None

        if arxAssign_extra_filters is not None:
            arxAssign_extra_filters.append(date_filter)
            arxAssign_final_filters = " AND ".join(arxAssign_extra_filters)
        else:
            arxAssign_final_filters = date_filter  # possible need change date

        arxAssign_selected_fields = [
            "AsnID",
            "CUSTCDI",
            "ContNum",
            "AsnStartDT",
            "AsnEndDT",
            "AsnType",
            "UserID",
            "businessdate",
        ]

        arxAssign = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_arxAssign",
            arxAssign_final_filters,
            arxAssign_selected_fields,
        )

        # ================== arxAgent ==================

        arxAgent_extra_filters = None

        if arxAgent_extra_filters is not None:
            arxAgent_extra_filters.append(date_filter)
            arxAgent_final_filters = " AND ".join(arxAgent_extra_filters)
        else:
            arxAgent_final_filters = date_filter  # possible need change date

        arxAgent_selected_fields = [
            "UserID",
            "UserName",
        ]

        arxAgent = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_arxAgent",
            arxAgent_final_filters,
            arxAgent_selected_fields,
        )

        arxAgent = arxAgent.withColumnRenamed("UserID", "UserIDArxAgent")

        arxAssign_arxAgent_join_condition = (
            arxAssign["UserID"] == arxAgent["UserIDArxAgent"]
        )

        arxAssign_final_portfolio_join_condition = (
            substring(arxAssign["CUSTCDI"], 1, 3) == portfolio["Prefix"]
        )

        target_table_arotron = arxAssign.join(
            portfolio,
            arxAssign_final_portfolio_join_condition,
            "left",
        ).join(
            arxAgent,
            arxAssign_arxAgent_join_condition,
            "left",
        )

        # HashKeys
        target_table_arotron_AssignmentId_primary_key_fields = [
            arxAssign.AsnID,
            lit("arotron"),
        ]

        target_table_arotron = target_table_arotron.withColumn(
            "AssignmentId",
            sha2(
                concat_ws(";", *target_table_arotron_AssignmentId_primary_key_fields),
                256,
            ),
        )

        # Add the new columns based on conditions, without using col()
        target_table_arotron = (
            target_table_arotron.withColumn(
                "ExternalCompanyFullName",
                when(target_table_arotron["AsnType"] == "I", None).otherwise(
                    target_table_arotron["UserName"]
                ),
            )
            .withColumn(
                "AgentFullName",
                when(target_table_arotron["AsnType"] == "E", None).otherwise(
                    target_table_arotron["UserName"]
                ),
            )
            .withColumn(
                "ExternalCompanyPartyKey",
                when(target_table_arotron["AsnType"] == "I", None).otherwise(
                    target_table_arotron["UserID"]
                ),
            )
            .withColumn(
                "AgentLoginName",
                when(target_table_arotron["AsnType"] == "E", None).otherwise(
                    target_table_arotron["UserID"]
                ),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.AssignmentId.alias("AssignmentId"),
            lit("arotron").cast("String").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale"))
            .cast("String")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            arxAssign.AsnID.cast("String").alias("AssignmentKey"),
            lit(None).cast("Int").alias("CaseCode"),
            lit(None).cast("String").alias("ExposureKey"),
            arxAssign.CUSTCDI.alias("PartyKey"),
            arxAssign.CUSTCDI.cast("String").alias("CDI"),
            lit(None).cast("String").alias("TaxIdentificationNumber"),
            arxAssign.ContNum.alias("AgreementKey"),
            lit(None).cast("String").alias("InternalAssignmentStartDate"),
            arxAssign.AsnStartDT.cast("String").alias("ExternalAssignmentStartDate"),
            arxAssign.AsnEndDT.cast("timestamp").alias("AssignmentRecallDate"),
            lit(None).cast("String").alias("ChannelCode"),
            lit(None).cast("String").alias("Channel"),
            lit(None).cast("String").alias("ExternalCompanyAgency"),
            target_table_arotron.ExternalCompanyPartyKey.cast("String").alias(
                "ExternalCompanyPartyKey"
            ),
            target_table_arotron.ExternalCompanyFullName.cast("String").alias(
                "ExternalCompanyFullName"
            ),
            arxAssign.AsnType.cast("String").alias("AssignmentTypeCode"),
            arxAssign.AsnType.cast("String").alias("AssignmentType"),
            lit(None).cast("String").alias("TeamCode"),
            lit(None).cast("String").alias("Team"),
            target_table_arotron.AgentLoginName.cast("String").alias("AgentPartyKey"),
            target_table_arotron.AgentFullName.cast("String").alias(
                "AgentFullName"
            ),  # change
            target_table_arotron.AgentLoginName.cast("String").alias("AgentLoginName"),
            lit(None).cast("String").alias("Supervisor"),
            lit(None).cast("String").alias("AssistantManager"),
            lit(None).cast("String").alias("Manager"),
            lit(None).cast("String").alias("Head"),
            lit(None).cast("Int").alias("AssignmentBucketCode"),
            lit(None).cast("String").alias("AssignmentBucket"),
            lit(None).cast("String").alias("AssignmentProcess"),
            lit(None).cast("String").alias("ProcessType"),
            lit(None).cast("Int").alias("PacketCode"),
            lit(None).cast("String").alias("Packet"),
            lit(None).cast("timestamp").alias("PacketPreviousEndDate"),
            lit(None).cast("timestamp").alias("PacketAssignmentDate"),
            lit(None).cast("Int").alias("PaccCode"),
            lit(None).cast("Int").alias("QueCode"),
            lit(None).cast("String").alias("Que"),
            lit(None).cast("String").alias("Segment"),
            lit(None).cast("String").alias("ExtraSegment"),
            lit(None).cast("Int").alias("StreamTypeCode"),
            lit(None).cast("String").alias("StreamType"),
            lit(None).cast("Int").alias("StreamStatusCode"),
            lit(None).cast("String").alias("StreamStatus"),
            lit(None).cast("Int").alias("StreamAssignedStatusCode"),
            lit(None).cast("String").alias("StreamAssignedStatus"),
            lit(None).cast("double").alias("OutstandingAmount"),
            lit(None).cast("double").alias("DebtAmount"),
            lit(None).cast("String").alias("CampaignName"),
            arxAssign.businessdate.cast("date").alias("BusinessDate"),
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"),   
        )


        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_assignment_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)